In [1]:
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

# 1. Initialize Embeddings (BGE-Base model)
model_name = "BAAI/bge-base-en-v1.5"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

# 2. Advanced Chunking (The Archon Method)
# This prevents legal sections from being cut in half.
def process_legal_docs(file_path):
    # Using 'elements' mode to maintain document structure
    loader = UnstructuredPDFLoader(file_path, mode="elements")
    docs = loader.load()
    
    # We use a smaller overlap but ensure we don't break mid-sentence
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", "Article", "Section", ".", " "]
    )
    chunks = text_splitter.split_documents(docs)
    return chunks

# 3. Vector Store Initialization & Ingestion
# Use a relative path so the DB is created in your project folder
db_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vector_database" 

# UPDATE: Set this to the exact name of the PDF you have uploaded
legal_pdf_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data\Mrs_Vandana_Dhirani_vs_Mrs_Arti_Kirloskar_on_27_July_2023.PDF" 

# Check if database already exists; if not, create and populate it
if not os.path.exists(db_path):
    print("Database not found. Starting ingestion...")
    if os.path.exists(legal_pdf_path):
        chunks = process_legal_docs(legal_pdf_path)
        # CRITICAL: This step moves data from PDF to ChromaDB
        vector_store = Chroma.from_documents(
            documents=chunks, 
            embedding_function=embeddings, 
            persist_directory=db_path
        )
        print(f"Ingested {len(chunks)} chunks successfully.")
    else:
        print(f"ERROR: Could not find {legal_pdf_path}. Please check the filename.")
        # Initialize an empty store to prevent downstream variable errors
        vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)
else:
    vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)
    print("Loaded existing database.")

# 4. DIAGNOSTIC: Verify if the database actually has data
# If this number is 0, your RAG will return "I cannot find this" for every query.
count = vector_store._collection.count()
print(f"Total document chunks in vector store: {count}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\user\AppData\Local\Temp\ipykernel_8960\2619900982.py:58: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)


Loaded existing database.
Total document chunks in vector store: 0


In [2]:
from sentence_transformers import CrossEncoder
import numpy as np

# Load Re-ranker (Archon Phase 1.2)
re_ranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def legal_research_tool(query: str):
    # Retrieve more results than needed (Top 20)
    initial_docs = vector_store.similarity_search(query, k=20)
    
    if not initial_docs:
        return "No relevant legal context found."

    # Re-ranking Logic
    doc_texts = [doc.page_content for doc in initial_docs]
    scores = re_ranker.predict([(query, doc) for doc in doc_texts])
    
    # Sort by score and take top 5
    ranked_indices = np.argsort(scores)[::-1][:5]
    top_docs = [doc_texts[i] for i in ranked_indices]
    
    return "\n---\n".join(top_docs)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

# State Definition
class AgentState(TypedDict):
    query: str
    research_results: str
    analysis: str
    final_response: str
    iteration_count: int

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# NODE 1: The Researcher (Now with verification)
def researcher_node(state: AgentState):
    query = state["query"]
    results = legal_research_tool(query)
    
    # If results are too short, try a broader search
    if len(results) < 100 and state.get("iteration_count", 0) < 1:
        broader_query = f"Legal principles related to {query}"
        results = legal_research_tool(broader_query)
    
    return {"research_results": results, "iteration_count": state.get("iteration_count", 0) + 1}

# NODE 2: The Strategist (Analysis)
def strategist_node(state: AgentState):
    context = state["research_results"]
    query = state["query"]
    
    prompt = f"""Analyze this legal context for the query: {query}
    Context: {context}
    Identify specific Articles, Case Names, or Dates. If not present, state 'DATA_MISSING'."""
    
    response = llm.invoke(prompt)
    return {"analysis": response.content}

# NODE 3: The Advisor (Final Response)
def advisor_node(state: AgentState):
    prompt = f"""You are a senior legal analyst. 
    Analyze the provided context to answer the query: {state['query']}
    
    Analysis: {state['analysis']}
    Context: {state['research_results']}
    
    INSTRUCTIONS:
    1. If the exact answer isn't there, look for similar concepts (e.g., cases involving the same court or similar legal principles).
    2. If you find a partial match, explain what you found.
    3. Only say 'I cannot find this' if there is absolutely zero relevant information."""
    
    response = llm.invoke(prompt)
    return {"final_response": response.content}

# Build the Graph
workflow = StateGraph(AgentState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("strategist", strategist_node)
workflow.add_node("advisor", advisor_node)

workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "strategist")
workflow.add_edge("strategist", "advisor")
workflow.add_edge("advisor", END)

app = workflow.compile()

In [4]:
test_queries = [
    {"q": "What happened in the case of Navjeet Budhiraja?", "key": "dismissed"},
    {"q": "What is the penalty under Article 254?", "key": "repugnancy"}
]

for i, test in enumerate(test_queries):
    inputs = {"query": test["q"], "iteration_count": 0}
    result = app.invoke(inputs)
    answer = result["final_response"]
    target = test["key"]
    
    # Improved Evaluation logic
    passed = target.lower() in answer.lower()
    
    # Handling your specific synonym requirements
    if not passed:
        if "dismiss" in answer.lower() and "dismissed" in target.lower():
            status = "PASS (Synonym)"
        else:
            status = "FAIL"
    else:
        status = "PASS"
        
    print(f"TEST {i+1}: {status}")
    print(f"Q: {test['q']}")
    print(f"A: {answer[:150]}...")
    print("-" * 30)

TEST 1: FAIL
Q: What happened in the case of Navjeet Budhiraja?
A: Based on the provided analysis, it appears that there is no relevant legal context available to determine what happened in the case of Navjeet Budhira...
------------------------------
TEST 2: FAIL
Q: What is the penalty under Article 254?
A: Based on the provided context, I was unable to find any relevant information regarding Article 254, including the penalty associated with it. The cont...
------------------------------
